## Operating on Data in Pandas

Pandas inherits NumPy's ufuncs (element-wise math: `+`, `np.exp`, `np.sin`, etc.), 
but adds two behaviors NumPy doesn't have:

1. **Index preservation** — applying a ufunc keeps the row/column labels attached.
2. **Index alignment** — combining two Series/DataFrames automatically matches 
   entries up by their labels, not by position, filling gaps with `NaN`.

This is the direct payoff of Pandas being "NumPy + labels" — operations stay 
meaningful even when your data doesn't perfectly line up.

In [1]:
import numpy as np
import pandas as pd


In [2]:
rng = np.random.RandomState(42)
ser = pd.Series(rng.randint(0, 10, 4))
df = pd.DataFrame(rng.randint(0, 10, (3, 4)), columns=['A', 'B', 'C', 'D'])

np.exp(ser)          # index (0,1,2,3) stays attached to the output
np.sin(df * np.pi / 4)  # same idea, column labels stay attached

,A,B,C,D
0,-1.000000,7.071068e-01,1.000000,-1.000000e+00
1,-0.707107,1.224647e-16,0.707107,-7.071068e-01
2,-0.707107,1.000000e+00,-0.707107,1.224647e-16


## Index alignment — Series

When you combine two Series with different indices, Pandas takes the **union** 
of both indices. Any label missing from one side becomes `NaN` in the result — 
this is Pandas' way of marking missing data.

In [3]:
area = pd.Series({'Alaska': 1723337, 'Texas': 695662, 'California': 423967})
population = pd.Series({'California': 38332521, 'Texas': 26448193, 'New York': 19651127})

population / area
# Alaska        NaN   <- no population entry for Alaska
# California    90.41
# New York      NaN    <- no area entry for New York
# Texas         38.02

Alaska              NaN
California    90.413926
New York            NaN
Texas         38.018740
dtype: float64

In [8]:
A = pd.Series([2, 4, 6], index=[0, 1, 2])
B = pd.Series([1, 3, 5], index=[1, 2, 3])

A + B                    # 0: NaN, 3: NaN (default)
A.add(B, fill_value=0)    # 0: 2.0, 3: 5.0  -> missing side treated as 0 instead

0    2.0
1    5.0
2    9.0
3    5.0
dtype: float64

### But with div there is a problem : 
fill_value=0 is a reasonable choice for +/- (where "missing = contributes nothing" makes clean sense), but it's a risky default for division specifically, exactly because of this asymmetry — filling a missing denominator with 0 doesn't mean "assume zero effect," it means "divide by zero," which is a very different, often unintended outcome.

In [9]:
population.truediv(area,fill_value=0)

Alaska         0.000000
California    90.413926
New York            inf
Texas         38.018740
dtype: float64

**Operator → method mapping** (needed whenever you want a custom `fill_value`):

| Operator | Method |
|---|---|
| `+` | `.add()` |
| `-` | `.sub()`, `.subtract()` |
| `*` | `.mul()`, `.multiply()` |
| `/` | `.truediv()`, `.div()`, `.divide()` |
| `//` | `.floordiv()` |
| `%` | `.mod()` |
| `**` | `.pow()` |

# Handling missing data

Real-world data is rarely complete. Pandas represents missing values using two Python sentinels:\
**`None`** and **`np.nan`** — and treats them as nearly interchangeable.\
Understanding the difference between them (and why NaN specifically "infects" calculations) avoids a lot of silent, confusing bugs.

## None vs NaN — why two different sentinels exist

- **`None`** — a plain Python object, meaning "nothing here." Because it's a 
  generic object (not a number), any array containing it gets forced into 
  `dtype=object` — which loses NumPy's fast, compiled-code performance and 
  falls back to slow, Python-level operations.
- **`NaN`** — a special *floating-point* value (`Not a Number`), part of the 
  IEEE float standard. Arrays containing `NaN` stay `dtype=float64` — fast, 
  native NumPy performance is preserved.

**Key difference in behavior**: operations on `None` typically **raise an 
error** (undefined). Operations on `NaN` **never error** — they just quietly 
propagate `NaN` through anything they touch.

In [10]:
vals1 = np.array([1, None, 3, 4])
vals1.dtype        # dtype('O')  -> object, slow
# vals1.sum()        # TypeError -> can't add int + None

vals2 = np.array([1, np.nan, 3, 4])
vals2.dtype        # dtype('float64')  -> fast, native

1 + np.nan          # nan  -> no error, just spreads
0 * np.nan           # nan  -> even multiplying by 0 doesn't "cancel" NaN

nan

**"NaN is a data virus"** — any arithmetic touching a `NaN` produces `NaN`.\
This means aggregates like `.sum()`/`.min()`/`.max()` won't error, but will silently return `NaN` if even one value is missing — often not what you want.

In [11]:
vals2.sum(), vals2.min(), vals2.max()
# (nan, nan, nan)  -> technically "works," but useless

np.nansum(vals2), np.nanmin(vals2), np.nanmax(vals2)
# (8.0, 1.0, 4.0)  -> the nan-aware versions ignore missing values instead

(np.float64(8.0), np.float64(1.0), np.float64(4.0))

## How Pandas handles this — automatic type upcasting

Pandas treats `None` and `NaN` as interchangeable, converting between them 
automatically. If you introduce a missing value into an integer Series, 
Pandas **upcasts** the whole column to `float64` — because integers have no 
native "missing" representation, but floats do (`NaN`).

In [12]:
pd.Series([1, np.nan, 2, None])
# 0    1.0
# 1    NaN
# 2    2.0
# 3    NaN
# dtype: float64   -> None got silently converted to NaN

x = pd.Series(range(2), dtype=int)   # starts as int64
x[0] = None
x
# 0    NaN
# 1    1.0
# dtype: float64   -> whole column upcast from int to float, just to hold one NaN

0    NaN
1    1.0
dtype: float64

## Upcasting rules by type — worth having as a reference table

| Original type | What happens when NA is introduced | Sentinel used |
|---|---|---|
| `float` | no change | `np.nan` |
| `object` | no change | `None` or `np.nan` |
| `int` | **upcast to `float64`** | `np.nan` |
| `bool` | **upcast to `object`** | `None` or `np.nan` |

Also worth remembering: **strings are always stored as `object` dtype in 
Pandas** — this connects directly to why string columns don't get the 
fast/compact treatment numeric columns do.

## The four core methods for handling nulls

| Method | Purpose |
|---|---|
| `isnull()` | boolean mask — `True` where value is missing |
| `notnull()` | opposite of `isnull()` |
| `dropna()` | returns a filtered copy with nulls removed |
| `fillna()` | returns a copy with nulls replaced |

In [14]:
data = pd.Series([1, np.nan, 'hello', None])

data.isnull()
# 0    False
# 1     True
# 2    False
# 3     True

data[data.notnull()]     # boolean mask used directly as an index — filters to non-null rows
# 0        1
# 2    hello

0        1
2    hello
dtype: object

## Dropping nulls — Series vs DataFrame

On a **Series**, `dropna()` is simple — just removes missing entries.

On a **DataFrame**, you can't drop a single cell — only a whole **row** or 
**column** — so `dropna()` needs extra parameters to control exactly how 
aggressive it is.

In [15]:
data.dropna()      # Series — drops missing entries directly

df = pd.DataFrame([[1, np.nan, 2],
                    [2, 3, 5],
                    [np.nan, 4, 6]])

df.dropna()                    # default: drop any ROW containing a null
df.dropna(axis='columns')       # drop any COLUMN containing a null instead

,2
0,2
1,5
2,6


## Filling nulls instead of dropping

`fillna()` replaces missing values rather than removing them — useful when 
dropping would lose too much real data alongside the missing entries.

In [20]:
data = pd.Series([1, np.nan, 2, None, 3], index=list('abcde'))

data.fillna(0)                     # fill with a fixed value

data.ffill()         # forward-fill: carry the PREVIOUS value forward


a    1.0
b    1.0
c    2.0
d    2.0
e    3.0
dtype: float64

In [19]:
data.bfill()          # back-fill: carry the NEXT value backward

a    1.0
b    2.0
c    2.0
d    3.0
e    3.0
dtype: float64

## Why this matters for energy data specifically

Sensor data (PV output, demand meters) commonly has gaps — a logger drops 
offline for an hour, a meter misreports. The choice between `dropna()` and 
`fillna()` (and *which* fill strategy) is a real modeling decision, not just 
cleanup: dropping a missing hour shortens your time series and can misalign 
it against other data; `ffill`/`bfill` assumes conditions didn't change much 
in the gap (reasonable for slow-changing values, risky for volatile ones 
like solar output on a partly cloudy day). Worth treating this as a 
deliberate choice per-dataset, not a default reflex.